In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.base import clone
from sklearn.ensemble import GradientBoostingRegressor

In [2]:
PROJECT_DIR = Path.cwd().resolve()

In [3]:
train = pd.read_csv(PROJECT_DIR / "data" / "train-test.csv")
validation = pd.read_csv(PROJECT_DIR / "data" / "validation.csv")

In [4]:
print("Train shape:", train.shape)
print("Validation shape:", validation.shape)

Train shape: (48000, 14)
Validation shape: (12000, 13)


In [5]:
print("Train data head:")
train.head()

Train data head:


,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,date,market_index,quote_signal,posted_rate
0,TR-000001,Richmond,Baltimore,38.09122,-76.78906,38.16908,-72.74564,274.3,Dry Van,30658.0,2025-01-01,0.95684,2.39595,645.41
1,TR-000002,Richmond,Philadelphia,38.09122,-76.78906,39.22317,-72.96710,280.5,Reefer,17555.0,2025-01-01,0.97623,2.43355,679.97
2,TR-000003,Philadelphia,Green Bay,39.22317,-72.96710,44.30296,-87.52871,967.8,Dry Van,31721.0,2025-01-01,1.00971,1.84491,1802.54
3,TR-000004,Hartford,Atlanta,39.55328,-72.18051,34.84933,-86.28940,965.4,Dry Van,32333.0,2025-01-01,0.94518,1.87712,1827.28
4,TR-000005,Dallas,Nashville,31.83025,-94.38343,35.29479,-88.08915,541.9,Reefer,35183.0,2025-01-01,0.98480,2.56300,1380.28


In [6]:
print("Validation data head:")
validation.head()

Validation data head:


,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,date,market_index,quote_signal
0,TE-000001,Baton Rouge,Mobile,30.50134,-92.65374,31.80442,-88.25195,331.4,Flatbed,19958.0,2025-11-01,0.90402,1.86388
1,TE-000002,Savannah,Los Angeles,33.60973,-82.29994,28.56624,-116.70249,2406.2,Reefer,34114.0,2025-11-01,0.92851,1.86240
2,TE-000003,San Francisco,Raleigh,35.19670,-121.69849,35.37376,-77.47605,2846.6,Dry Van,33435.0,2025-11-01,0.88117,1.95480
3,TE-000004,Dayton,Los Angeles,39.87206,-85.62931,28.56624,-116.70249,2261.6,Dry Van,25392.0,2025-11-01,0.89156,2.14579
4,TE-000005,Memphis,San Antonio,35.56802,-89.52871,29.50969,-98.40059,800.4,Flatbed,NaN,2025-11-01,0.88308,2.22541


In [7]:
print("\nColumn names:", train.columns.tolist())


Column names: ['load_id', 'pickup', 'delivery', 'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon', 'distance', 'equipment', 'weight', 'date', 'market_index', 'quote_signal', 'posted_rate']


In [8]:
print("Cols type and missing vals:", train.info())
print("\nMissing vals per column:", train.isna().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48000 entries, 0 to 47999
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   load_id       48000 non-null  object 
 1   pickup        48000 non-null  object 
 2   delivery      48000 non-null  object 
 3   pickup_lat    48000 non-null  float64
 4   pickup_lon    48000 non-null  float64
 5   delivery_lat  48000 non-null  float64
 6   delivery_lon  48000 non-null  float64
 7   distance      48000 non-null  float64
 8   equipment     48000 non-null  object 
 9   weight        47700 non-null  float64
 10  date          48000 non-null  object 
 11  market_index  47626 non-null  float64
 12  quote_signal  48000 non-null  float64
 13  posted_rate   48000 non-null  float64
dtypes: float64(9), object(5)
memory usage: 5.1+ MB
Cols type and missing vals: None

Missing vals per column: load_id           0
pickup            0
delivery          0
pickup_lat        0
pickup_lon  

In [9]:
print("Numerical summary:")
train.describe()

Numerical summary:


,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,weight,market_index,quote_signal,posted_rate
count,48000.000000,48000.000000,48000.000000,48000.000000,48000.000000,47700.000000,47626.000000,48000.000000,48000.000000
mean,35.647545,-90.928964,35.641175,-90.857310,1135.856654,31028.844004,1.083387,2.062468,2373.980682
std,4.315285,13.482431,4.317199,13.476589,728.564416,9391.440620,0.168091,0.291391,1486.493245
min,28.357650,-121.698490,28.357650,-121.698490,70.000000,-47500.000000,0.676390,0.692280,57.220000
25%,31.986910,-98.400590,31.986910,-98.400590,550.400000,25800.000000,0.949670,1.891030,1251.555000
50%,35.294790,-88.089150,35.294790,-87.528710,953.300000,31436.500000,1.055800,2.055750,2030.760000
75%,39.411040,-83.285060,39.411040,-83.285060,1645.525000,37018.000000,1.219590,2.221685,3330.750000
max,44.302960,-69.500000,44.302960,-69.500000,3439.800000,47500.000000,1.467780,3.610350,25533.000000


In [10]:
print("Equipment types:", train["equipment"].value_counts())
print("\nDuplicate load ID:", train["load_id"].duplicated().sum())
print("\nNon positive weights:", train[train["weight"] <= 0].shape[0])
print("\nDate range:")
for name, frame in [("Train", train), ("Validation", validation)]:
    print(f"{name}: {frame['date'].min()} to {frame['date'].max()}")
    print(f"Invalid dates: {frame['date'].isna().sum()}")

Equipment types: equipment
Dry Van    27202
Reefer     12045
Flatbed     8753
Name: count, dtype: int64

Duplicate load ID: 0

Non positive weights: 292

Date range:
Train: 2025-01-01 to 2025-10-31
Invalid dates: 0
Validation: 2025-11-01 to 2025-12-31
Invalid dates: 0


In [11]:
cleaned_train = train.copy()
cleaned_train.loc[cleaned_train["weight"] <= 0, "weight"] = float("nan")

In [12]:
print("\nMissing weights after cleaning:", cleaned_train["weight"].isna().sum())


Missing weights after cleaning: 592


In [13]:
print("Distance, weight and posted rate summary:")
print(cleaned_train[["distance", "weight", "posted_rate"]].describe())

Distance, weight and posted rate summary:
           distance        weight   posted_rate
count  48000.000000  47408.000000  48000.000000
mean    1135.856654  31415.358674   2373.980682
std      728.564416   7994.902979   1486.493245
min       70.000000   5000.000000     57.220000
25%      550.400000  25922.000000   1251.555000
50%      953.300000  31493.500000   2030.760000
75%     1645.525000  37063.000000   3330.750000
max     3439.800000  47500.000000  25533.000000


In [14]:
print("Posted rate percentiles:")
print(cleaned_train["posted_rate"].quantile([0.01,0.5,0.95,0.99,1.0]))

print("\nNon positive distances:", (cleaned_train["distance"] <= 0).sum())
print("Non positive posted rates:", (cleaned_train["posted_rate"] <= 0).sum())


Posted rate percentiles:
0.01      327.1688
0.50     2030.7600
0.95     4953.7665
0.99     5972.8340
1.00    25533.0000
Name: posted_rate, dtype: float64

Non positive distances: 0
Non positive posted rates: 0


In [15]:
cleaned_train["date"] = pd.to_datetime(cleaned_train["date"], errors="coerce")
train_part = cleaned_train[cleaned_train["date"] < "2025-09-01"].copy()
dev_part = cleaned_train[(cleaned_train["date"] >= "2025-09-01") & (cleaned_train["date"] < "2025-10-01")].copy()
test_part = cleaned_train[cleaned_train["date"] >= "2025-10-01"].copy()

In [16]:
print("Chronological split")
for name, frame in [("Train", train_part), ("Dev", dev_part), ("Test", test_part)]:
    print(f"{name}: {frame.shape[0]} rows, {frame['date'].min()} to {frame['date'].max()}")
assert len(train_part) + len(dev_part) + len(test_part) == len(cleaned_train), "Split sizes do not match original data size"

Chronological split
Train: 38477 rows, 2025-01-01 00:00:00 to 2025-08-31 00:00:00
Dev: 4670 rows, 2025-09-01 00:00:00 to 2025-09-30 00:00:00
Test: 4853 rows, 2025-10-01 00:00:00 to 2025-10-31 00:00:00


In [17]:
baseline_rate = train_part["posted_rate"].median()
baseline_predictions = [baseline_rate] * len(dev_part)
baseline_mae = mean_absolute_error(dev_part["posted_rate"], baseline_predictions)
print("Baseline predicted price: $", baseline_rate)
print("September baseline MAE: $", baseline_mae)

Baseline predicted price: $ 2026.04
September baseline MAE: $ 1151.1144625267668


In [18]:
numerical_features = ["distance", "weight"]
categorical_features = ["pickup", "delivery", "equipment"]
features = numerical_features + categorical_features

In [19]:
X_train = train_part[features]
y_train = train_part["posted_rate"]
X_dev = dev_part[features]
y_dev = dev_part["posted_rate"]

In [20]:
numeric_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median", add_indicator=True))])
categorical_pipeline = Pipeline([("encoder", OneHotEncoder(handle_unknown="ignore"))])
preprocessor = ColumnTransformer([("numeric", numeric_pipeline, numerical_features), ("categorical", categorical_pipeline, categorical_features)])

In [21]:
model = Pipeline([("preprocessor", preprocessor), ("regressor", RandomForestRegressor(n_estimators=100, random_state=42))])
model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(add_indicator=True,
                                                                                 strategy='median'))]),
                                                  ['distance', 'weight']),
                                                 ('categorical',
                                                  Pipeline(steps=[('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['pickup', 'delivery',
                                                   'equipment'])])),
                ('regressor', RandomForestRegressor(random_state=42))])

In [22]:
dev_predictions = model.predict(X_dev)
model_mae = mean_absolute_error(y_dev, dev_predictions)
improvement = 100 * (baseline_mae - model_mae) / baseline_mae
print("Baseline September MAE: $", baseline_mae)
print("Random Forest September MAE: $", model_mae)
print("Improvement over baseline: ", improvement, "%")

Baseline September MAE: $ 1151.1144625267668
Random Forest September MAE: $ 154.91955528907923
Improvement over baseline:  86.54177665798575 %


In [23]:
errors = pd.DataFrame({"actual": y_dev, "predicted": dev_predictions})
errors["absolute_error"] = (errors["actual"] - errors["predicted"]).abs()
print("Absolute errors percentiles:")
print(errors["absolute_error"].quantile([0.5,0.90,0.95,0.99,1.0]))
print("\nTop largest errors:")
print(errors.nlargest(10, "absolute_error"))

Absolute errors percentiles:
0.50       48.650350
0.90      213.951970
0.95      399.299505
0.99     2411.382492
1.00    16258.896800
Name: absolute_error, dtype: float64

Top largest errors:
         actual  predicted  absolute_error
40096  20132.37  3873.4732      16258.8968
40171  20361.89  4439.3755      15922.5145
40322  14949.25  3257.7779      11691.4721
42149  13026.22  2834.2571      10191.9629
42735  12797.54  3600.9369       9196.6031
42881  13503.63  4418.0800       9085.5500
39362  11684.70  2894.6595       8790.0405
40158  12240.81  3738.3590       8502.4510
38771  11377.49  3485.8954       7891.5946
42645  13420.59  5606.2617       7814.3283


In [24]:
boosting_model = Pipeline([("preprocessor", clone(preprocessor)), ("regressor", GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=4, min_samples_leaf=20, random_state=42, loss="absolute_error"))])
boosting_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(add_indicator=True,
                                                                                 strategy='median'))]),
                                                  ['distance', 'weight']),
                                                 ('categorical',
                                                  Pipeline(steps=[('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['pickup', 'delivery',
                                                   'equipment'])])),
                ('regressor',
                 GradientBoostingRegressor(learning_rate=0.05,
                                           loss='absolute_error', max_depth=4,
                                           min_samples_leaf=20,
                                           n_estimators=200,
                                           random_state=42))])

In [25]:
boosting_predictions = boosting_model.predict(X_dev)
boosting_mae = mean_absolute_error(y_dev, boosting_predictions)
print("Baseline September MAE: $", baseline_mae)
print("Random Forest September MAE: $", model_mae )
print("Gradient Boosting September MAE: $", boosting_mae)   

Baseline September MAE: $ 1151.1144625267668
Random Forest September MAE: $ 154.91955528907923
Gradient Boosting September MAE: $ 110.86862093920926


In [26]:
def build_features(frame):
    result= frame[features].copy()
    dates = pd.to_datetime(frame["date"], errors="coerce")

    result["day_of_week"] = dates.dt.dayofweek
    result["is_weekend"] = dates.dt.dayofweek >= 5

    day_of_year = dates.dt.dayofyear
    result["year_sin"] = np.sin(2 * np.pi * day_of_year / 365.25)
    result["year_cos"] = np.cos(2 * np.pi * day_of_year / 365.25)
    return result   

In [27]:
date_numeric_features = numerical_features + ["day_of_week", "is_weekend", "year_sin", "year_cos"]
date_preprocessor = ColumnTransformer([("numeric", clone(numeric_pipeline), date_numeric_features), ("categorical", clone(categorical_pipeline), categorical_features)])

In [28]:
date_model = Pipeline([("preprocessor", date_preprocessor), ("regressor", clone(boosting_model.named_steps["regressor"]))])

X_train_date = build_features(train_part)
X_dev_date = build_features(dev_part)

date_model.fit(X_train_date, y_train)

date_predictions = date_model.predict(X_dev_date)
date_mae = mean_absolute_error(y_dev, date_predictions)

print("Boosting without dates MAE: $", boosting_mae)
print("Boosting with dates MAE: $", date_mae)

Boosting without dates MAE: $ 110.86862093920926
Boosting with dates MAE: $ 118.41886566366549


In [29]:
final_train_part = pd.concat([train_part, dev_part], ignore_index=True)
final_model = clone(boosting_model)
final_model.fit(final_train_part[features], final_train_part["posted_rate"])


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(add_indicator=True,
                                                                                 strategy='median'))]),
                                                  ['distance', 'weight']),
                                                 ('categorical',
                                                  Pipeline(steps=[('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['pickup', 'delivery',
                                                   'equipment'])])),
                ('regressor',
                 GradientBoostingRegressor(learning_rate=0.05,
                                           loss='absolute_error', max_depth=4,
                                           min_samples_leaf=20,
                                           n_estimators=200,
                                           random_state=42))])

In [30]:
test_predictions = final_model.predict(test_part[features])
test_mae = mean_absolute_error(test_part["posted_rate"], test_predictions)
print("October test MAE: $", test_mae)

October test MAE: $ 115.76415433544834


In [31]:
final_model = clone(boosting_model)
final_model.fit(cleaned_train[features], cleaned_train["posted_rate"])

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(add_indicator=True,
                                                                                 strategy='median'))]),
                                                  ['distance', 'weight']),
                                                 ('categorical',
                                                  Pipeline(steps=[('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['pickup', 'delivery',
                                                   'equipment'])])),
                ('regressor',
                 GradientBoostingRegressor(learning_rate=0.05,
                                           loss='absolute_error', max_depth=4,
                                           min_samples_leaf=20,
                                           n_estimators=200,
                                           random_state=42))])

In [32]:
clean_validation = validation.copy()
clean_validation.loc[clean_validation["weight"] <= 0, "weight"] = float("nan")
validation_predictions = final_model.predict(clean_validation[features])
assert np.isfinite(validation_predictions).all()
assert (validation_predictions > 0).all()

In [33]:
template = pd.read_csv(PROJECT_DIR / "data" / "validation-predictions-template.csv")
assert len(clean_validation) == 12000
assert clean_validation["load_id"].is_unique
assert len(template) == 12000
assert template["load_id"].is_unique
assert set(template["load_id"]) == set(clean_validation["load_id"])


In [36]:
predictions_by_id = pd.Series(validation_predictions, index=clean_validation["load_id"])
template["predicted_rate"] = template["load_id"].map(predictions_by_id)
assert template["predicted_rate"].notna().all()

In [37]:
final = template[[ "load_id", "predicted_rate" ]]
final.to_csv(PROJECT_DIR / "data" / "validation_predictions.csv", index=False)
print(final.head())
print(f"Saved {len(final)} predictions ")

     load_id  predicted_rate
0  TE-000001      825.269878
1  TE-000002     5012.805578
2  TE-000003     5111.005459
3  TE-000004     4059.855615
4  TE-000005     1821.223108
Saved 12000 predictions 


In [38]:
december_path = PROJECT_DIR / "data" / "december-chart-inputs.csv"
december = pd.read_csv(december_path)
december["predicted_rate"] = final_model.predict(december[features])

assert len(december) == 31
assert np.isfinite(december["predicted_rate"]).all()
assert (december["predicted_rate"] > 0).all()

In [39]:
december.to_csv(december_path, index=False)
print(december[["date", "predicted_rate"]].head())
print(f"Saved {len(december)} December predictions")

        date  predicted_rate
0  12/1/2025      834.965242
1  12/2/2025      834.965242
2  12/3/2025      834.965242
3  12/4/2025      834.965242
4  12/5/2025      834.965242
Saved 31 December predictions
